In [42]:
import pandas as pd
import numpy as np

Adjust the Waiting Time Component of EACH ROW and See how Many Fall Out of the 20-minute Criteria
- Push back tap-in and tap-out data 
- Analyse effect of varying delay durations (e.g., 5, 10, and 15 minutes) to determine if adjustments to the transfer window are necessary

In [18]:
df4 = pd.read_pickle('../data/df4.pkl')

In [19]:
delay_minutes = [5, 10, 15]

In [20]:
df_sim = df4.copy()

In [44]:
df4['PATRON_CATG_DESC_TXT']

0          Student
1          Student
2          Student
3          Student
4            Adult
            ...   
7698506      Adult
7698507      Adult
7698508      Adult
7698509      Adult
7698510      Adult
Name: PATRON_CATG_DESC_TXT, Length: 7698511, dtype: str

In [23]:
#make sure this matches that in the classifier
df_sim['waiting_time_allowed'] = 20
df_sim['extras_allowed'] = 0
df_sim['Total_allowance'] = df_sim['waiting_time_allowed'] + df_sim['extras_allowed']

In [52]:
for delay in delay_minutes:
    sim_col    = f'waiting_time_plus_extra_{delay}'
    flag_col   = f'exceeds_time_allowance_{delay}'       # temporal only
    final_flag = f'final_termination_flag_{delay}'       # combined all 3
    seq_col    = f'final_full_journey_seq_{delay}'       # journey seq from combined flag

    # 1. shift waiting time by delay
    df_sim[sim_col] = df_sim['waiting_time_plus_extra'] + delay

    # 2. recompute temporal flag (mirrors your exact np.select logic)
    df_sim[f'temporal_flag_reason_{delay}'] = np.select(
        [
            df_sim['time_gap_mins'].isna(),
            df_sim['predicted_walking_time_mins'].isna(),
            df_sim[sim_col].isna(),
            df_sim[sim_col] > df_sim['Total_allowance'],
        ],
        [
            'null_time_gap',
            'null_predicted_walking_time',
            'null_waiting_time_plus_extra',
            'exceeds_total_allowance',
        ],
        default='within_total_allowance'
    )
    df_sim[flag_col] = df_sim[f'temporal_flag_reason_{delay}'] != 'within_total_allowance'

    # 3. recombine with binary + spatial (these are unchanged by delay)
    df_sim[final_flag] = (
        df_sim['journey_termination_flag']  # binary
        | df_sim[flag_col]                  # simulated temporal
        | df_sim['spatial_break_flag_final'] # spatial
    )

    # 4. recompute journey sequence from combined flag
    shifted = df_sim.groupby('CRD_NUM')[final_flag].shift(fill_value=False)
    df_sim[seq_col] = shifted.groupby(df_sim['CRD_NUM']).cumsum()

    # 5. summary
    original_journeys  = df_sim.groupby('CRD_NUM')['final_full_journey_seq'].max().sum()
    simulated_journeys = df_sim.groupby('CRD_NUM')[seq_col].max().sum()

    print(f"\n--- Delay +{delay} minutes ---")
    print(f"Original total journeys:  {original_journeys:,.0f}")
    print(f"Simulated total journeys: {simulated_journeys:,.0f}")
    print(f"Additional journeys created: {simulated_journeys - original_journeys:,.0f}")


--- Delay +5 minutes ---
Original total journeys:  3,500,684
Simulated total journeys: 3,914,608
Additional journeys created: 413,924

--- Delay +10 minutes ---
Original total journeys:  3,500,684
Simulated total journeys: 4,023,338
Additional journeys created: 522,654

--- Delay +15 minutes ---
Original total journeys:  3,500,684
Simulated total journeys: 4,250,816
Additional journeys created: 750,132


In [53]:
#which patrons are most affected?
patron_delay_summary = []

for delay in delay_minutes:
    flag_col = f'exceeds_time_allowance_{delay}'
    
    newly_flagged = (df_sim[flag_col] == True) & (df_sim['exceeds_time_allowance'] == False)
    
    counts = (
        df_sim.groupby('PATRON_CATG_DESC_TXT')
        .apply(lambda grp: pd.Series({
            f'newly_flagged_{delay}min': newly_flagged[grp.index].sum(),
            'total_rides': len(grp),
            f'pct_affected_{delay}min': newly_flagged[grp.index].sum() / len(grp) * 100
        }))
        .reset_index()
    )
    patron_delay_summary.append(counts[['PATRON_CATG_DESC_TXT', f'newly_flagged_{delay}min', f'pct_affected_{delay}min']])

# merge all delays, keep total_rides from first summary
total_rides_patron = (
    df_sim.groupby('PATRON_CATG_DESC_TXT')
    .size()
    .reset_index(name='total_rides')
)

from functools import reduce
patron_result = reduce(
    lambda l, r: l.merge(r, on='PATRON_CATG_DESC_TXT', how='outer'),
    patron_delay_summary
)
patron_result = patron_result.merge(total_rides_patron, on='PATRON_CATG_DESC_TXT', how='left')
patron_result = patron_result.fillna(0).sort_values('pct_affected_5min', ascending=False)

print(patron_result[[
    'PATRON_CATG_DESC_TXT', 'total_rides',
    'newly_flagged_5min',  'pct_affected_5min',
    'newly_flagged_10min', 'pct_affected_10min',
    'newly_flagged_15min', 'pct_affected_15min',
]])


  PATRON_CATG_DESC_TXT  total_rides  newly_flagged_5min  pct_affected_5min  \
1       Senior Citizen      1265163             27439.0           2.168811   
0                Adult      5339213             76558.0           1.433882   
2              Student       835069              9075.0           1.086737   

   newly_flagged_10min  pct_affected_10min  newly_flagged_15min  \
1              71249.0            5.631606             156029.0   
0             209665.0            3.926890             480036.0   
2              27123.0            3.247995              71634.0   

   pct_affected_15min  
1           12.332719  
0            8.990763  
2            8.578213  


In [54]:
df_sim['entry_hour'] = pd.to_datetime(df_sim['ENTRY_TM'], format='%H:%M:%S').dt.hour

entry_delay_summary = []

for delay in delay_minutes:
    flag_col = f'exceeds_time_allowance_{delay}'
    
    newly_flagged = (df_sim[flag_col] == True) & (df_sim['exceeds_time_allowance'] == False)
    
    counts = (
        df_sim.groupby('entry_hour')
        .apply(lambda grp: pd.Series({
            f'newly_flagged_{delay}min': newly_flagged[grp.index].sum(),
            f'pct_affected_{delay}min': newly_flagged[grp.index].sum() / len(grp) * 100
        }))
        .reset_index()
    )
    entry_delay_summary.append(counts)

total_rides_hour = (
    df_sim.groupby('entry_hour')
    .size()
    .reset_index(name='total_rides')
)

entry_result = reduce(
    lambda l, r: l.merge(r, on='entry_hour', how='outer'),
    entry_delay_summary
)
entry_result = entry_result.merge(total_rides_hour, on='entry_hour', how='left')
entry_result = entry_result.fillna(0).sort_values('entry_hour')

print(entry_result[[
    'entry_hour', 'total_rides',
    'newly_flagged_5min',  'pct_affected_5min',
    'newly_flagged_10min', 'pct_affected_10min',
    'newly_flagged_15min', 'pct_affected_15min',
]])


    entry_hour  total_rides  newly_flagged_5min  pct_affected_5min  \
0            0        16660                98.0           0.588235   
1            1          926                 0.0           0.000000   
2            2          252                 0.0           0.000000   
3            3          628                 0.0           0.000000   
4            4          713                26.0           3.646564   
5            5        77642              1248.0           1.607377   
6            6       416893              4094.0           0.982027   
7            7       725423              5843.0           0.805461   
8            8       694170              5751.0           0.828471   
9            9       405099              4600.0           1.135525   
10          10       288635              5048.0           1.748922   
11          11       308698              5931.0           1.921295   
12          12       331573              6634.0           2.000766   
13          13      

In [55]:
for delay in delay_minutes:
    pct_col = f'pct_affected_{delay}min'
    flagged_col = f'newly_flagged_{delay}min'
    
    # filter out hours with very few rides to avoid misleading %
    meaningful = entry_result[entry_result['total_rides'] >= 100]
    
    top3 = meaningful.nlargest(3, pct_col)[['entry_hour', 'total_rides', flagged_col, pct_col]]
    bot3 = meaningful.nsmallest(3, pct_col)[['entry_hour', 'total_rides', flagged_col, pct_col]]
    
    print(f"\n{'='*50}")
    print(f"Delay +{delay} minutes")
    print(f"{'='*50}")
    print(f"\nTop 3 most affected hours:")
    print(top3.to_string(index=False))
    print(f"\nTop 3 least affected hours:")
    print(bot3.to_string(index=False))


Delay +5 minutes

Top 3 most affected hours:
 entry_hour  total_rides  newly_flagged_5min  pct_affected_5min
          4          713                26.0           3.646564
         16       423926              8951.0           2.111453
         12       331573              6634.0           2.000766

Top 3 least affected hours:
 entry_hour  total_rides  newly_flagged_5min  pct_affected_5min
          1          926                 0.0                0.0
          2          252                 0.0                0.0
          3          628                 0.0                0.0

Delay +10 minutes

Top 3 most affected hours:
 entry_hour  total_rides  newly_flagged_10min  pct_affected_10min
          4          713                 73.0           10.238429
         16       423926              22283.0            5.256342
          5        77642               3874.0            4.989568

Top 3 least affected hours:
 entry_hour  total_rides  newly_flagged_10min  pct_affected_10min
       

In [56]:
# if needed
#delays = pd.read_csv('../data/delays.csv')

Use instances of real delays?

Identify regions/MRT lines that would benefit the most?